# Gold 3_14 — Customer Engagement Versions (SCD2)

One row per lead per **engagement version** — a new version opens only when the engagement hash changes. Complements 3_15 (daily snapshots = trends) with a true change log (versions = what changed, and when).

* **Population**: all Coaching Client leads from `gold.health_metrics` (intern clients included — currently 1,371).
* **Hash** (opens a version on change): LEAD_ID, LEAD_EMAIL, LEAD_NAME, CONTRACT_VALUE, SENTIMENTS_LAST_30_DAYS, SLACK_ENGAGED_FLAG, PLATFORM_ENGAGED_FLAG, MEETING_ENGAGED_FLAG, ENGAGEMENT_PATTERN, BASE_ENGAGEMENT_SCORE, FINAL_SCORE, HEALTH_BAND.
* **VRSN_START_DATE** = run date the version opened (first run = today). **VRSN_END_DATE** = run date the hash changed (the version was superseded); NULL while current.
* Clock metrics (DAYS_SINCE_*, TOTAL_MISSED_CALLS, CONTRACT_START_DATE) are **not hashed** — they refresh on the open version every run; closed versions keep their as-of values.
* TOTAL_MISSED_CALLS is the trailing **14-day window** count, not a lifetime total. It still influences versions through FINAL_SCORE / HEALTH_BAND; only score-neutral call changes (e.g. 1→2) do not open one.
* A lead that leaves the Coaching Client population gets its open version closed that run; if it returns, a new version opens.
* SOURCE_UPDATED_AT = the CRM lead record's `date_updated`. INSERT_DATE = when this pipeline created the version row.
* Scoring identical to 3_14 (spec 5.1): base from pattern (90, 80, 75, 60, 50, 40, 30, 20, or no-data), FINAL_SCORE = base + missed-call adjustment + sentiment adjustment — no adjustments on NO DATA, not capped.

In [0]:
%sql
-- Engagement source: the 3_14 scoring logic applied to the FULL health_metrics
-- population (intern clients included — this is the versioned record of everyone),
-- plus the change-detection hash. Hash = engagement state + identity + contract
-- value; clock metrics deliberately excluded (they drift daily and would
-- degenerate SCD2 into a daily snapshot — 3_15 already owns that job).
CREATE OR REPLACE TEMP VIEW customer_engagement_source AS (
WITH base AS (
    SELECT
        h.lead_id                                                          AS LEAD_ID,
        h.lead_name                                                        AS LEAD_NAME,
        h.primary_customer_email                                           AS LEAD_EMAIL,
        TRY_CAST(REGEXP_REPLACE(h.Contracted_Value, '[^0-9.]', '') AS DOUBLE) AS CONTRACT_VALUE,
        -- Contract start: CRM custom field first (reference source, 1,323/1,371
        -- clients), sale-activity date as fallback for leads the field misses.
        COALESCE(TRY_CAST(h.contract_start_date AS DATE),
                 TRY_CAST(h.date_of_sale AS DATE))                  AS CONTRACT_START_DATE,
        h.days_since_last_email                                            AS DAYS_SINCE_LAST_EMAIL,
        h.days_since_last_login                                            AS DAYS_SINCE_LAST_LOGIN,
        h.days_since_last_meeting                                          AS DAYS_SINCE_LAST_MEETING,
        h.days_since_last_message_from_client                              AS DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT,
        h.days_since_last_message_from_team_member                         AS DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
        h.missed_calls_14_days                                             AS TOTAL_MISSED_CALLS,
        COALESCE(UPPER(TRIM(h.sentiment)), 'NO_SENTIMENT')                 AS SENTIMENTS_LAST_30_DAYS,
        TRY_CAST(h.date_updated AS TIMESTAMP)                              AS SOURCE_UPDATED_AT,
        CASE WHEN h.days_since_last_message_from_client IS NULL THEN NULL
             WHEN h.days_since_last_message_from_client <= 14 THEN 1
             ELSE 0 END                                                    AS SLACK_ENGAGED_FLAG,
        CASE WHEN h.days_since_last_login IS NULL THEN NULL
             WHEN h.days_since_last_login <= 14 THEN 1
             ELSE 0 END                                                    AS PLATFORM_ENGAGED_FLAG,
        CASE WHEN h.days_since_last_meeting IS NULL OR h.days_since_last_meeting_calendly IS NULL THEN NULL
             WHEN h.days_since_last_meeting <= 30 OR h.days_since_last_meeting_calendly <= 30 THEN 1
             ELSE 0 END                                                    AS MEETING_ENGAGED_FLAG,
        CASE WHEN h.missed_calls_14_days = 0 THEN 5
             WHEN h.missed_calls_14_days BETWEEN 1 AND 2 THEN 0
             WHEN h.missed_calls_14_days >= 3 THEN -15 END                 AS missed_call_adjustment,
        CASE WHEN UPPER(TRIM(h.sentiment)) = 'POSITIVE' THEN 10
             WHEN UPPER(TRIM(h.sentiment)) = 'NEUTRAL'  THEN 5
             WHEN UPPER(TRIM(h.sentiment)) = 'NEGATIVE' THEN -25
             ELSE 0 END                                                    AS sentiment_adjustment
    FROM crm_ingestion.gold.health_metrics h
),
patterns AS (
    SELECT
        b.*,
        CASE
            WHEN b.SLACK_ENGAGED_FLAG = 1 AND b.PLATFORM_ENGAGED_FLAG = 1 AND b.MEETING_ENGAGED_FLAG = 1 THEN 'SLACK + PLATFORM + MEETING'
            WHEN b.SLACK_ENGAGED_FLAG = 1 AND b.PLATFORM_ENGAGED_FLAG = 1 AND (b.MEETING_ENGAGED_FLAG = 0 OR b.MEETING_ENGAGED_FLAG IS NULL) THEN 'SLACK + PLATFORM'
            WHEN (b.SLACK_ENGAGED_FLAG = 0 OR b.SLACK_ENGAGED_FLAG IS NULL) AND b.PLATFORM_ENGAGED_FLAG = 1 AND b.MEETING_ENGAGED_FLAG = 1 THEN 'PLATFORM + MEETING'
            WHEN b.SLACK_ENGAGED_FLAG = 1 AND (b.PLATFORM_ENGAGED_FLAG = 0 OR b.PLATFORM_ENGAGED_FLAG IS NULL) AND b.MEETING_ENGAGED_FLAG = 1 THEN 'SLACK + MEETING'
            WHEN b.SLACK_ENGAGED_FLAG = 1 AND (b.PLATFORM_ENGAGED_FLAG = 0 OR b.PLATFORM_ENGAGED_FLAG IS NULL) AND (b.MEETING_ENGAGED_FLAG = 0 OR b.MEETING_ENGAGED_FLAG IS NULL) THEN 'SLACK ONLY'
            WHEN (b.SLACK_ENGAGED_FLAG = 0 OR b.SLACK_ENGAGED_FLAG IS NULL) AND b.PLATFORM_ENGAGED_FLAG = 1 AND (b.MEETING_ENGAGED_FLAG = 0 OR b.MEETING_ENGAGED_FLAG IS NULL) THEN 'PLATFORM ONLY'
            WHEN (b.SLACK_ENGAGED_FLAG = 0 OR b.SLACK_ENGAGED_FLAG IS NULL) AND (b.PLATFORM_ENGAGED_FLAG = 0 OR b.PLATFORM_ENGAGED_FLAG IS NULL) AND b.MEETING_ENGAGED_FLAG = 1 THEN 'MEETING ONLY'
            WHEN b.SLACK_ENGAGED_FLAG = 0 AND b.PLATFORM_ENGAGED_FLAG = 0 AND b.MEETING_ENGAGED_FLAG = 0 THEN 'NO ENGAGEMENT'
            WHEN b.SLACK_ENGAGED_FLAG = 0 AND b.PLATFORM_ENGAGED_FLAG = 0 AND b.MEETING_ENGAGED_FLAG IS NULL THEN 'NO ENGAGEMENT'
            WHEN b.SLACK_ENGAGED_FLAG IS NULL AND b.PLATFORM_ENGAGED_FLAG = 0 AND b.MEETING_ENGAGED_FLAG = 0 THEN 'NO ENGAGEMENT'
            WHEN b.SLACK_ENGAGED_FLAG = 0 AND b.PLATFORM_ENGAGED_FLAG IS NULL AND b.MEETING_ENGAGED_FLAG = 0 THEN 'NO ENGAGEMENT'
            ELSE 'NO DATA'
        END AS ENGAGEMENT_PATTERN
    FROM base b
),
scored AS (
    SELECT
        p.*,
        CASE p.ENGAGEMENT_PATTERN
            WHEN 'SLACK + PLATFORM + MEETING' THEN 90
            WHEN 'SLACK + PLATFORM'         THEN 80
            WHEN 'PLATFORM + MEETING'       THEN 75
            WHEN 'SLACK + MEETING'          THEN 60
            WHEN 'PLATFORM ONLY'            THEN 50
            WHEN 'MEETING ONLY'            THEN 40
            WHEN 'SLACK ONLY'              THEN 30
            WHEN 'NO ENGAGEMENT'           THEN 20
            ELSE NULL
        END AS BASE_ENGAGEMENT_SCORE
    FROM patterns p
),
finals AS (
    SELECT
        s.*,
        CASE WHEN s.ENGAGEMENT_PATTERN = 'NO DATA' THEN NULL
             ELSE s.BASE_ENGAGEMENT_SCORE + s.missed_call_adjustment + s.sentiment_adjustment
        END AS FINAL_SCORE,
        CASE WHEN s.ENGAGEMENT_PATTERN = 'NO DATA' THEN 'NO DATA'
             WHEN s.BASE_ENGAGEMENT_SCORE + s.missed_call_adjustment + s.sentiment_adjustment >= 75 THEN 'GOOD'
             WHEN s.BASE_ENGAGEMENT_SCORE + s.missed_call_adjustment + s.sentiment_adjustment >= 50 THEN 'AVERAGE'
             ELSE 'POOR'
        END AS HEALTH_BAND
    FROM scored s
)
SELECT
    f.LEAD_ID, f.LEAD_EMAIL, f.LEAD_NAME,
    f.CONTRACT_VALUE, f.CONTRACT_START_DATE,
    f.DAYS_SINCE_LAST_EMAIL, f.DAYS_SINCE_LAST_LOGIN, f.DAYS_SINCE_LAST_MEETING,
    f.DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT, f.DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
    f.TOTAL_MISSED_CALLS, f.SENTIMENTS_LAST_30_DAYS,
    f.SLACK_ENGAGED_FLAG, f.PLATFORM_ENGAGED_FLAG, f.MEETING_ENGAGED_FLAG,
    f.ENGAGEMENT_PATTERN, f.BASE_ENGAGEMENT_SCORE, f.FINAL_SCORE, f.HEALTH_BAND,
    f.SOURCE_UPDATED_AT,
    MD5(TO_JSON(STRUCT(
        f.LEAD_ID, f.LEAD_EMAIL, f.LEAD_NAME,
        f.CONTRACT_VALUE, f.SENTIMENTS_LAST_30_DAYS,
        f.SLACK_ENGAGED_FLAG, f.PLATFORM_ENGAGED_FLAG, f.MEETING_ENGAGED_FLAG,
        f.ENGAGEMENT_PATTERN, f.BASE_ENGAGEMENT_SCORE, f.FINAL_SCORE, f.HEALTH_BAND
    ))) AS HASH_VALUE
FROM finals f
)

In [0]:
%sql
-- Bootstrap: create the SCD2 target only if missing. On nightly job runs this is
-- a no-op — the three SCD2 steps below do all the work.
CREATE TABLE IF NOT EXISTS crm_ingestion.gold.customer_health_profile (
    LEAD_ID                             STRING NOT NULL,
    LEAD_EMAIL                          STRING,
    LEAD_NAME                           STRING,
    VRSN_START_DATE                     DATE    COMMENT 'Run date this version opened (first run = first appearance).',
    VRSN_END_DATE                       DATE    COMMENT 'Run date the hash changed and this version was superseded. NULL = current version.',
    CONTRACT_VALUE                      DOUBLE,
    CONTRACT_START_DATE                 DATE,
    DAYS_SINCE_LAST_EMAIL               INT,
    DAYS_SINCE_LAST_LOGIN               INT,
    DAYS_SINCE_LAST_MEETING             INT,
    DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT      BIGINT,
    DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER BIGINT,
    TOTAL_MISSED_CALLS                  BIGINT  COMMENT 'Missed inbound calls in the trailing 14-day window (not a lifetime total).',
    SENTIMENTS_LAST_30_DAYS             STRING,
    SLACK_ENGAGED_FLAG                  INT,
    PLATFORM_ENGAGED_FLAG               INT,
    MEETING_ENGAGED_FLAG                INT,
    ENGAGEMENT_PATTERN                  STRING,
    BASE_ENGAGEMENT_SCORE               INT     COMMENT 'From the pattern table (90, 80, 75, 60, 50, 40, 30, 20, or no-data).',
    FINAL_SCORE                        INT     COMMENT 'Base + missed-call adjustment + sentiment adjustment. No adjustments on NO DATA. Not capped.',
    HEALTH_BAND                         STRING,
    HASH_VALUE                          STRING  COMMENT 'MD5 over the 13 engagement-hash columns; a change opens a new version.',
    SOURCE_UPDATED_AT                   TIMESTAMP COMMENT 'CRM date_updated of the lead record when this version was created.',
    INSERT_DATE                         TIMESTAMP COMMENT 'When this pipeline created the version row.'
);

COMMENT ON TABLE crm_ingestion.gold.customer_health_profile IS
    'SCD2 engagement history: one row per lead per engagement version. A new version opens when the engagement hash changes; VRSN_END_DATE NULL = current. Companion to customer_health_history (daily snapshots).';

In [0]:
%sql
-- SCD2 step 1: close versions whose hash changed, refresh clock metrics on
-- unchanged open versions. Delta evaluates the MATCHED clauses in order.
MERGE INTO crm_ingestion.gold.customer_health_profile AS t
USING customer_engagement_source AS s
ON  t.LEAD_ID = s.LEAD_ID
AND t.VRSN_END_DATE IS NULL
WHEN MATCHED AND t.HASH_VALUE <> s.HASH_VALUE THEN
    UPDATE SET t.VRSN_END_DATE = CURRENT_DATE()
WHEN MATCHED THEN
    UPDATE SET
        t.CONTRACT_START_DATE                 = s.CONTRACT_START_DATE,
        t.DAYS_SINCE_LAST_EMAIL               = s.DAYS_SINCE_LAST_EMAIL,
        t.DAYS_SINCE_LAST_LOGIN               = s.DAYS_SINCE_LAST_LOGIN,
        t.DAYS_SINCE_LAST_MEETING             = s.DAYS_SINCE_LAST_MEETING,
        t.DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT = s.DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT,
        t.DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER = s.DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
        t.TOTAL_MISSED_CALLS                  = s.TOTAL_MISSED_CALLS,
        t.SOURCE_UPDATED_AT                   = s.SOURCE_UPDATED_AT

In [0]:
%sql
-- SCD2 step 2: open a new version for new leads and for leads whose version
-- step 1 just closed. Unchanged leads still have an open version -> skipped.
INSERT INTO crm_ingestion.gold.customer_health_profile (
    LEAD_ID, LEAD_EMAIL, LEAD_NAME, VRSN_START_DATE, VRSN_END_DATE,
    CONTRACT_VALUE, CONTRACT_START_DATE,
    DAYS_SINCE_LAST_EMAIL, DAYS_SINCE_LAST_LOGIN, DAYS_SINCE_LAST_MEETING,
    DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT, DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
    TOTAL_MISSED_CALLS, SENTIMENTS_LAST_30_DAYS,
    SLACK_ENGAGED_FLAG, PLATFORM_ENGAGED_FLAG, MEETING_ENGAGED_FLAG,
    ENGAGEMENT_PATTERN, BASE_ENGAGEMENT_SCORE, FINAL_SCORE, HEALTH_BAND,
    HASH_VALUE, SOURCE_UPDATED_AT, INSERT_DATE
)
SELECT
    s.LEAD_ID, s.LEAD_EMAIL, s.LEAD_NAME,
    CURRENT_DATE()                    AS VRSN_START_DATE,
    CAST(NULL AS DATE)               AS VRSN_END_DATE,
    s.CONTRACT_VALUE, s.CONTRACT_START_DATE,
    s.DAYS_SINCE_LAST_EMAIL, s.DAYS_SINCE_LAST_LOGIN, s.DAYS_SINCE_LAST_MEETING,
    s.DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT, s.DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
    s.TOTAL_MISSED_CALLS, s.SENTIMENTS_LAST_30_DAYS,
    s.SLACK_ENGAGED_FLAG, s.PLATFORM_ENGAGED_FLAG, s.MEETING_ENGAGED_FLAG,
    s.ENGAGEMENT_PATTERN, s.BASE_ENGAGEMENT_SCORE, s.FINAL_SCORE, s.HEALTH_BAND,
    s.HASH_VALUE, s.SOURCE_UPDATED_AT,
    CURRENT_TIMESTAMP()               AS INSERT_DATE
FROM customer_engagement_source s
WHERE NOT EXISTS (
    SELECT 1
    FROM crm_ingestion.gold.customer_health_profile v
    WHERE v.LEAD_ID = s.LEAD_ID
      AND v.VRSN_END_DATE IS NULL
)

In [0]:
%sql
-- SCD2 step 3: a lead that left the Coaching Client population closes its open
-- version today. If it returns later, step 2 opens a fresh version.
UPDATE crm_ingestion.gold.customer_health_profile
SET VRSN_END_DATE = CURRENT_DATE()
WHERE VRSN_END_DATE IS NULL
  AND LEAD_ID NOT IN (SELECT LEAD_ID FROM customer_engagement_source)

In [0]:
%sql
-- Validation 1: open versions must equal the current population (1,339),
-- and every open version started on the first run date.
SELECT
    COUNT(*)            AS open_versions,
    COUNT(DISTINCT LEAD_ID) AS distinct_leads,
    MIN(VRSN_START_DATE)    AS first_version_date,
    MAX(VRSN_START_DATE)    AS latest_version_start
FROM crm_ingestion.gold.customer_health_profile
WHERE VRSN_END_DATE IS NULL

In [0]:
%sql
-- Validation 2: current engagement mix by band (open versions only)
SELECT HEALTH_BAND, ENGAGEMENT_PATTERN, COUNT(*) AS leads
FROM crm_ingestion.gold.customer_health_profile
WHERE VRSN_END_DATE IS NULL
GROUP BY 1, 2
ORDER BY leads DESC

In [0]:
%sql
-- VALIDATION — what changed today: for every lead whose version closed today,
-- compare the outgoing (o) vs incoming (n) version. This is the change log behind
-- the version churn validation above, with focus on what the 3_09 login fix moved.
SELECT
    COUNT(*)                                              AS versions_opened_today,
    SUM(CASE WHEN o.PLATFORM_ENGAGED_FLAG IS NULL AND n.PLATFORM_ENGAGED_FLAG IS NOT NULL THEN 1 ELSE 0 END) AS platform_gained_value,
    SUM(CASE WHEN o.PLATFORM_ENGAGED_FLAG IS NULL AND n.PLATFORM_ENGAGED_FLAG = 1 THEN 1 ELSE 0 END) AS platform_gained_engaged,
    SUM(CASE WHEN o.PLATFORM_ENGAGED_FLAG IS NULL AND n.PLATFORM_ENGAGED_FLAG = 0 THEN 1 ELSE 0 END) AS platform_gained_disengaged,
    SUM(CASE WHEN o.PLATFORM_ENGAGED_FLAG = 1 AND n.PLATFORM_ENGAGED_FLAG = 0 THEN 1 ELSE 0 END) AS platform_aged_out,
    SUM(CASE WHEN o.ENGAGEMENT_PATTERN <> n.ENGAGEMENT_PATTERN THEN 1 ELSE 0 END) AS pattern_changed,
    SUM(CASE WHEN o.ENGAGEMENT_PATTERN = 'NO DATA' AND n.ENGAGEMENT_PATTERN <> 'NO DATA' THEN 1 ELSE 0 END) AS left_no_data
FROM crm_ingestion.gold.customer_health_profile n
JOIN crm_ingestion.gold.customer_health_profile o
  ON  o.LEAD_ID = n.LEAD_ID
 AND o.VRSN_END_DATE = CURRENT_DATE()
 AND n.VRSN_START_DATE = CURRENT_DATE()

In [0]:
# # Reference sample 2 — 51 leads pasted verbatim (VRSN window 2026-11-01 -> 2026-11-02).
# # Wrapped as a string with the standard header so the comparison cell can parse it.
# teacher_txt51 = """LEAD_ID,LEAD_EMAIL,LEAD_NAME,VRSN_START_DATE,VRSN_END_DATE,CONTRACT_VALUE,CONTRACT_START_DATE,DAYS_SINCE_LAST_EMAIL,DAYS_SINCE_LAST_LOGIN,DAYS_SINCE_LAST_MEETING,DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT,DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,TOTAL_MISSED_CALLS,SENTIMENTS_LAST_30_DAYS,SLACK_ENGAGED_FLAG,PLATFORM_ENGAGED_FLAG,MEETING_ENGAGED_FLAG,ENGAGEMENT_PATTERN,BASE_ENGAGEMENT_SCORE,FINAL_SCORE,HEALTH_BAND,HASH_VALUE,SOURCE_UPDATED_AT,INSERT_DATE
# lead_ChP5jERqnkknMWsmk7cjjU95DwVQcsBElLNQAhHIeRC,rkangadi@gmail.com,REVANTH KUMAR ANGADI / revanth,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,12000,2025-02-05,400,231,257,235,184,0," Positive  ",0,0,0,NO ENGAGEMENT,20,35,POOR,8dcc3d1b5d3f295e03d1e63384d9d996,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_qqsqjXCHJ8GFXhmPgVEXq8RaS1wMv0zASFUl7YniVv6,mobilesnipe@yahoo.com / cameyer02@gmail.com / chris.a.meyer@outlook.com,Chris / Chris Meyer,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,8000,2026-06-03,104,34,111,44,11,0," Neutral  ",0,0,0,NO ENGAGEMENT,20,30,POOR,f624dd812dc0fe1b08855e4e59bea848,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_udIurVBKIReifXffGFZ2JSofxPpqPjjKbexzFQYjEze,srichakra1008@gmail.com,Srinivasu Pappula,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,10000,2024-12-28,642,636,673,331,81,0," Positive  ",0,0,,NO DATA,9999,9999,NO DATA,dce14c37392cf6f9d3b12b9deaa08e63,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_u6IZlTk9nqseo2Of01A8Y7NYeXIjnqfUO6ASTJeO5hK,lancegoinsteam@gmail.com / lgoins02@icloud.com,Lance Goins / Lance Goins Jr,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,15000,2025-02-15,41,273,9,12,10,0," Positive  ",1,0,1,SLACK + MEETING,60,75,GOOD,41e73b619460248ac7b706b3b3925e72,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_T7cxMCuCdwZhY6uZ7KbSApRf6X62T9t6yllqEXvy4y0,invi.presavi@gmail.com,Santosh Kothapally,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,,2024-11-19,531,121,223,219,212,0," Neutral  ",0,0,0,NO ENGAGEMENT,20,30,POOR,1a001a54adbf94f4bf0d685570cbbf31,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_A9atbEd6hCFGejqvjrkWJrBUq6usZWfzDa2F2B27Sn4,thomasmf2005@gmail.com,Thomas Franklin,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,21750,2025-01-29,221,24,292,317,219,0," Positive  ",0,0,0,NO ENGAGEMENT,20,35,POOR,d99c0482e00ad2c80f7e6641c17d77e6,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_jjXijFVi7LzKgD03dzVYTPVhOZ5TPte9TZ7oc5U12s4,chaowilliamthao@gmail.com,Chao William Thao,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,13000,2025-09-17,4,2,2,85,36,0," Negative  ",0,1,1,PLATFORM + MEETING,75,55,AVERAGE,e2b244353c7c6dbababd115b23464874,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_4M9Ak2AUihd4EtCRxJwJkg0E2wBThlWg3KNzEhVOVkc,richheineman@gmail.com,Richard Heineman,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,13000,2024-11-19,674,653,684,497,107,0," Positive  ",0,0,,NO DATA,9999,9999,NO DATA,6025522affda69a7522e864f4dab8758,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_GgKSCFJouyqDcvTjUzyAtgFsvmW7OnQrYpimNshb2uR,leegramirez@gmail.com,Lee Ramirez,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,11000,2024-10-04,201,292,230,,,0,NO_SENTIMENT,,0,0,NO DATA,9999,9999,NO DATA,a13005c829da69590f240a2187c2089f,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_rBSwCGFGNGMZrUChNuzEMiyVMIcN4jKb09o6yutzyjh,barro8444@gmail.com,Allen Barro / Allen,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,20500,2026-03-03,236,185,27,57,12,0," Neutral  ",0,0,1,MEETING ONLY,40,50,AVERAGE,c7158590ed3244dcae14a92df81ba068,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_3sihoxH8ciWXQwDA04uw52mSWiCs8dE2jMRDm047sZp,shaungray03@gmail.com,Shaun,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,10000,2024-11-19,790,810,753,213,36,0," Neutral  ",0,0,,NO DATA,9999,9999,NO DATA,f78859c36f97536d6e2c7c7827dbe7e2,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_dm3TA1CSHlYxu77uh9IrQtewcB85SnmLqt2Od1DFXxV,terrytlin25@gmail.com / terrylinpi@gmail.com,Terry Lin,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,20000,2025-12-01,16,95,16,5,4,0," Neutral  ",1,0,1,SLACK + MEETING,60,70,AVERAGE,89d1239a6d5b5a71b9e9cf3a4bcf0e1e,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_wzHeXuRk7mSoRP3nEuJWFsKssIEL9goh42UCgLZNE4Z,aaron.dorboryan@gmail.com,Aaron B Dorboryan / Aaron Dorboryan,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,22500,2026-09-02,51,54,60,26,39,0," Positive  ",0,0,0,NO ENGAGEMENT,20,35,POOR,753194a8b066c638482d543ec448fc25,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_tCtoegcSHM8UUkzigI0Buf5bOgj0Xc9bhM4dEYnZ8Vm,krazymatty@gmail.com / phil@theprintshopwy.com,Philip Matthews,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,12000,2024-12-14,319,684,706,196,129,0," Positive  ",0,0,,NO DATA,9999,9999,NO DATA,c3d922c9be1ccd40484ed2b52203c3e8,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_BuRBSpv90vtwOXRC5QFWGRzxCY8C1YNd3V0dIuoNpF6,hubiarnatarau@gmail.com,Aliaksandr Hubiarnatarau,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,,2024-11-19,723,501,950,,,0,NO_SENTIMENT,,0,,NO DATA,9999,9999,NO DATA,f76f8cd23aa5a56211af94049fe75301,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_SlXpBiUWdkvxjSoi6Q4uHOWeUOdA74pIxf1jw92UxBY,ahdanstp@gmail.com,Ahdan St. Phard,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,3500,2024-11-19,681,474,521,329,12,0," Positive  ",0,0,,NO DATA,9999,9999,NO DATA,1c2cebd7fd1efe7c75e8570afd50f8e8,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_3CD0rSlS3wehHHMMpcOzIHkdFVox9py7FtOk2cO6bm1,kkellyannw@gmail.com,Kelly Wiedl,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,20000,2026-02-08,9,3,9,52,125,0," Positive  ",0,1,1,PLATFORM + MEETING,75,90,GOOD,8a37b5a86fb04c4ab8c1b38c89adecf4,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_S0Gtl4HxYRcwteuuWtSQkzPYTAgGWcT9B8H9DjLT1rf,rbradleywatts@gmail.com,Brad Watts,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,8000,2025-03-28,18,8,19,2,2,0," Positive  ",1,1,1,SLACK + PLATFORM + MEETING,90,105,GOOD,15aa1ccb052b865a20b4873cb052c792,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_xW0ujxCdGWiS9hEze41a1sd7oKOMIygwUKKUu69n1kN,cedeno_mark@yahoo.com,cedeno_mark@yahoo.com cedeno_mark@yahoo.com / Mark Cedeno / Mark,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,20000,2025-11-19,268,5,44,43,12,0," Neutral  ",0,1,0,PLATFORM ONLY,50,60,AVERAGE,b26fb1c5d8185304b02f0051e07742d7,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_irAVgQamcsWlipDr3xehdEUZ3HCPtcmcCHiTSYy4Y94,arpitshah9815@gmail.com / arpitshah1101@gmail.com,arpitshah1101@gmail.com arpitshah1101@gmail.com / Arpit Shah,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,10000,2024-11-19,626,451,626,331,128,0," Neutral  ",0,0,,NO DATA,9999,9999,NO DATA,dacf9ffe218d305424f334adc5855188,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_2KYzgx1W4P74MUuAmTJ6mfkXcPa21bJMOSoaT1GBtTS,mohammadbkhurshid@gmail.com,Mohammad Khurshid,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,20000,2026-03-27,58,0,50,3,4,0," Positive  ",1,1,1,SLACK + PLATFORM + MEETING,90,105,GOOD,82689257d6477b0131a6b2c9342d9a1b,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_DEuKjBQafjDcSY3VTIGVpTdNk2wf3hIWNQw9TgzzDtE,prmpnchl@gmail.com,prmpnchl@gmail.com prmpnchl@gmail.com / Prem Panchal,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,25000,2025-10-07,153,41,82,47,12,0," Positive  ",0,0,0,NO ENGAGEMENT,20,35,POOR,a6f49c2fde3d2a1e2ee4d611f43451ea,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_lvmz9xJ3U62ORQXHkMnMmHksO6FVwhbGVF6c84F10pS,josetijero10@gmail.com,Joseph Tijero,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,15000,2026-05-29,32,63,90,9,9,0," Positive  ",1,0,0,SLACK ONLY,30,45,POOR,63059c9a065aeb20474ddc95a2b9657b,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_M6mLRxj7dzdbqho2MTzp7duOhK7fnYmFx7Q3qt1ZCbh,damorrisx77@gmail.com,Darien Morris,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,12000,2025-02-24,159,266,166,177,80,0," Positive  ",0,0,0,NO ENGAGEMENT,20,35,POOR,0c2e59009a2716b8e3be92c696b0d149,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_k1Y1JXhbWzyf2o76bK2CpkzolskUyjJdjR6UQVraInY,delores.yates@yahoo.com,Delores Yates,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,18000,2025-09-19,345,41,45,284,12,0," Neutral  ",0,0,0,NO ENGAGEMENT,20,30,POOR,c76b84d52878f5b66f2142dff45f1de6,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_F5PqwgUMikrHTjXElHtzK6tgNj8sCPWel5ys587w0R8,mbenn7963@gmail.com,Michael Bennett,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,15000,2026-03-18,102,7,108,72,11,0," No Student Activity  ",0,1,0,PLATFORM ONLY,50,55,AVERAGE,cd6d61f44af4a2ebf25240ebcc3cc860,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_hxXIrpOnHJTuKlbBrde1fCUW3AOCG4jdt3q48o1ztos,hemendra0329@gmail.com,Hemendra Gummadi,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,,2024-11-19,762,122,446,,,0,NO_SENTIMENT,,0,,NO DATA,9999,9999,NO DATA,4afbf1537c0b47865bdebb31c623f053,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_TVWzbZsH1J0phQOE5Zg6oPepLLLNK5JHGmb4xsDu791,nidal.alkhatib1989@gmail.com,Alhasan / Alhasan N Alkhatib / alhasan alkhatib,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,10000,2025-08-18,60,12,37,30,67,0," Neutral  ",0,1,0,PLATFORM ONLY,50,60,AVERAGE,e9988fc3882dc0efee68c1a7970137e6,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_nW4ZUW2N1CDARpjxqQDDFwZQDlPEbS2l5esqp9rW6z7,j_hippolyte@hotmail.com,James Hippolyte,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,,2024-11-19,215,185,199,206,13,0," Neutral  ",0,0,0,NO ENGAGEMENT,20,30,POOR,f5f5d85dc9388d006d94af7b26c34f6a,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_ejAU2nuz0vRJ9t6BAGi75nbj3maXp7AR2b9LXhvDWVi,bondhsu@yahoo.com,Bond Hsu,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,8000,2024-11-19,506,499,502,,,0,NO_SENTIMENT,,0,,NO DATA,9999,9999,NO DATA,49d1a99760c185c185747c2d7bc5e61c,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_SNHukEeNPaXdQNxL8XoUAXmv1yR7cQCpTxWbTI11EEn,jayanthkuchipudi1@gmail.com,Jayanth kuchipudi / Jayanth Kuchipudi,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,18000,2026-03-20,48,61,78,47,45,0," Neutral  ",0,0,0,NO ENGAGEMENT,20,30,POOR,51d4bdc9c5702daf7e6f6d99db776619,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_vcb7pI3v2D7BdaX57qhCUxx6STt7sUeRULmbVGn3ev8,bob.frazer@gmail.com,Bob Frazer / Bob,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,22000,2026-04-04,4,84,4,89,12,0," Neutral  ",0,0,1,MEETING ONLY,40,50,AVERAGE,3168d0f3c167ff222ab898eb05eb838a,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_jFoUKkbW4Wwwxbn7BSGx21Xh6Y1X06C1wV0AM1IBK4V,sandeshunt@gmail.com,sandesh,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,10000,2024-11-19,489,590,489,219,0,0," Neutral  ",0,0,,NO DATA,9999,9999,NO DATA,1b394ea111874f67881664ef54c7f25b,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_gveNozbYXD7hSj1M4EES7Zsm0uDjED1ENkNbGTYy9K1,omemije7567@yahoo.com,Olivia Memije,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,30000,2025-11-05,74,18,5,1,3,0," Neutral  ",1,0,1,SLACK + MEETING,60,70,AVERAGE,fda1f3131fd5c803953aae731568d527,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_x3yGaMemCpXnEV0jJSF3l4ZRSkh0xZWnGWwLKEJVlbt,venkat.r.mummadi@gmail.com / venkat.mummadi001@gmail.com,Venkat Mummadi,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,10000,2024-11-19,65,80,76,75,36,0," Neutral  ",0,0,0,NO ENGAGEMENT,20,30,POOR,652ecc15a0fa7126263d062170234b97,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_wWNyE85iXfFrcBcAfpeqDReMQ7ZrU0MvaVGPTcLlR0o,priya.dalal.jobs@gmail.com,Priya Dalal,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,6000,2026-05-15,4,5,4,96,0,0," Positive  ",0,1,1,PLATFORM + MEETING,75,90,GOOD,3b338e06bc92dc96c72f6e6077789d82,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_u6uWBIyJqTX7QWgkXkIqCYI4218Q0Malww3lv1DFY4B,ahume9@gmail.com / 1aaronhume@gmail.com,Aaron Hume / Aaron,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,20500,2026-02-17,3,3,3,1,11,0," Positive  ",1,1,1,SLACK + PLATFORM + MEETING,90,105,GOOD,0be333ced16db4cba6f1c174090aa811,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_Y0fOpjgagdun3W75RDtjKEWQvBnTV7PbUO3mu3U7GOn,bborozny@gmail.com,William John Borozny / Bill Borozny,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,30000,2026-03-30,6,0,6,,,0,NO_SENTIMENT,,1,1,PLATFORM + MEETING,75,80,GOOD,ed7359507867bef9080798a7f9628c2e,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_KdYd9qddSn4I7yMiaHcIdVDPNGv0QxXSw84MLjtGHnl,patxnj@gmail.com,Nish Patel / patxnj@gmail.com / N P / Nish P,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,5000,2025-12-04,17,112,17,108,2,0," Neutral  ",0,0,1,MEETING ONLY,40,50,AVERAGE,8f9e5f7b211954acf1bb8ceecac438aa,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_9qDFg5x7hEXozfFE5JO6oJ6FYRGvexHv1V9hpd7tkqn,msgarris@gmail.com,Michael (Sean) Garrison / Michael Garrison,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,29000,2026-01-21,34,51,9,32,33,0," Neutral  ",0,0,1,MEETING ONLY,40,50,AVERAGE,490aed0250ba969a5ee07e82643f4c74,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_UvOWJp0lzjNg9jdP0VwY3VbwRN2W10nkzXkylcVmINv,carngwekwe@gmail.com,Carine C Ngwekwe / Carine Ngwekwe,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,8000,2025-07-31,18,11,20,78,36,0," Neutral  ",0,1,1,PLATFORM + MEETING,75,85,GOOD,5fd0b0d57ece7f5221ee93d6b627dc11,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_O9TanZcbpZXoXaWXtbCdeBjlmbtiNS7N8DIjEuGrZyD,mounchandini@yahoo.com / lucilemyriam@gmail.com,lucilemyriam@gmail.com / Myriam lucile Mounchandini / Myriam Mounchandini / Myriam,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,15000,2025-11-20,57,2,2,1,16,0," Neutral  ",1,1,1,SLACK + PLATFORM + MEETING,90,100,GOOD,eee411cd6b3536e01a4701c5a0f07d1a,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_QuDEUzaesEtmoAo8nXcZnYg1wnUmaC3WnGkvHEmdAtk,parikkapadia21@gmail.com,parik kapadia,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,8000,2024-11-19,16,320,16,,,0,NO_SENTIMENT,,0,1,MEETING ONLY,40,45,POOR,69cc8ad86deb517642232c12cef0daaa,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_NWh8fieoLIevXkASUJsBjlYl3RRFxiEIWCle1E0TjNv,t.shalounda@gmail.com / tshalounda@yahoo.com,Shalounda / Shalounda Thompson,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,12500,2025-02-24,164,632,642,552,480,0," Positive  ",0,0,0,NO ENGAGEMENT,20,35,POOR,c76a9486b1e5e15a908dfe5f568a7298,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_QsrviUPRux5OguKuh6CYKEPtFcbSuP90hsMkiKcWnsK,m.a.zeni@hotmail.com / aamirzeni@gmail.com,Mohammad Aamir Zeni / Aamir Zeni,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,13000,2025-03-10,144,1,145,95,96,0," Neutral  ",0,1,1,PLATFORM + MEETING,75,85,GOOD,3d997cb9a5977db837bf0f40c727264f,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_nT0uZoy2laPpO2ydjiL7bs8tMOr0Ee6d6SkWX9Cepqm,arkeetect@gmail.com,Ahmed Rahil Khan,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,15000,2025-02-05,26,60,22,39,16,0," Negative  ",0,0,1,MEETING ONLY,40,20,POOR,8df621fb1c8478ab292596522689d09e,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_Y1lFxpsYh1uJrNX4sAqzkYMDAH89O5XLPLlpeaPcfSm,nouraltoblany@gmail.com / nour@m2scontracting.com,Nour Toblany,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,8000,2024-11-19,2,283,2,1,24,0," Positive  ",1,0,1,SLACK + MEETING,60,75,GOOD,b1cd59fa662e66233fff9d1bb5d4df05,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_0yAAT8wIwq4hmMWVfceo3oagma5H41Pww3mXF22wtX9,adam.fury516@gmail.com,Adam Bey,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,25000,2026-08-10,12,6,2,4,3,0," Positive  ",1,1,1,SLACK + PLATFORM + MEETING,90,105,GOOD,721a1d028777ac91087cd9e111bf379b,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_UVx90yHW6aUREWDkNNEODiyyxsHcFZW3oOk8osJaIhl,rumuky@gmail.com / ritzyrita2001@yahoo.com,ritzyrita2001@yahoo.com / Rita Umuko,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,22500,2026-10-14,5,0,5,18,5,0," Positive  ",0,1,1,PLATFORM + MEETING,75,90,GOOD,419fe0fcf402ffa6b7aa3ba72a1d5455,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_WZWtAlzDcuprsBtdPJCcFpClf6E1dObdNo37YQ2yuZF,shasee1202@gmail.com / duverneau24@gmail.com / maria.bordyug@gmail.com,Maria / maria.bordyug@gmail.com maria.bordyug@gmail.com / Maria Bordyug,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,,2024-11-19,188,179,188,884,127,0," No Student Activity  ",0,0,0,NO ENGAGEMENT,20,25,POOR,8df0eab8ea3127156f3dd81281350ee3,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_GTMCd7t6fzIVX8jO419CjzHaVgT0zMNXcOfg10IV7zV,malik53188@gmail.com,Fakeha Malik,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,35000,2026-04-17,39,2,3,,,0,NO_SENTIMENT,,1,1,PLATFORM + MEETING,75,80,GOOD,30d44d2398dea7d499053b9ddee2e785,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# """


In [0]:
# import io
# import pandas as pd

# # Reference output pasted in the cell above, verbatim (VRSN window 2026-11-01 -> 2026-11-02)
# teacher_txt = """LEAD_ID,LEAD_EMAIL,LEAD_NAME,VRSN_START_DATE,VRSN_END_DATE,CONTRACT_VALUE,CONTRACT_START_DATE,DAYS_SINCE_LAST_EMAIL,DAYS_SINCE_LAST_LOGIN,DAYS_SINCE_LAST_MEETING,DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT,DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,TOTAL_MISSED_CALLS,SENTIMENTS_LAST_30_DAYS,SLACK_ENGAGED_FLAG,PLATFORM_ENGAGED_FLAG,MEETING_ENGAGED_FLAG,ENGAGEMENT_PATTERN,BASE_ENGAGEMENT_SCORE,FINAL_SCORE,HEALTH_BAND,HASH_VALUE,SOURCE_UPDATED_AT,INSERT_DATE
# lead_bo0Aw17YkLn795YW2j2XHTsh43KqkhqjFMjDG1UwH38,sydibuari@yahoo.com / sbuari88@gmail.com,Sydi Buari,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,10000,2024-12-26,363,623,370,330,213,0, Positive  ,0,0,,NO DATA,20,9999,NO DATA,c847cd44e3f4f3087bc12d9cd102e173,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_NEAsiuAhEAuKTWYXmB2c9EEcf7wLlSqRSjMKNX3xdGH,oyegunwaseyi@gmail.com,Seyi Oyegunwa,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,20000,2026-01-27,177,142,50,83,110,0, Positive  ,0,0,0,NO ENGAGEMENT,20,35,POOR,45570028e917ff4db7185bf4b00de031,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_C7o1N9SfcGAsLCqiHGxxeOmVoWQN7ShD9LzATOWcdcm,irron.williams@gmail.com,Irron Williams,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,7000,2025-05-30,375,231,343,259,221,0, Positive  ,0,0,,NO DATA,9999,9999,NO DATA,3fb3932d75a87a85dd67bb498ea894bf,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_Pvq800BVsA671hrI7S6hA4fbAD2dUOeWYJTjL4P8e9n,annambasit123@gmail.com,Annam Basit,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,10000,2025-01-11,529,330,445,333,12,0, Positive  ,0,0,,NO DATA,9999,9999,NO DATA,e2a9f37036151d4602266be5ffb923bd,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_9ppVFUPLYDFu7DboruWWUaqWDMq7piyhWxS4enw5oxy,suryadevarabasaveswaramukesh@gmail.com,Basaveswara Mukesh Suryadevara,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,8000,2026-06-13,33,14,29,9,0,0, Positive  ,1,1,1,SLACK + PLATFORM + MEETING,90,105,GOOD,49002d77f222db86f78033bb1367632a,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_8kvThfspxJJBRfsNxOYhVCuxsQWSZu8CJEF3wLWCX76,nicholo.neri@gmail.com,Nic Neri,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,16000,2025-11-20,230,135,19,215,21,0, No Student Activity  ,0,0,1,MEETING ONLY,40,45,POOR,dc5eaf6196e4405bbbd9f258b9a3856c,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_DBOhRgpYGb8kIQXypKsKdxxTy97QY0kJ8CBYsmmL4SG,attribution.matters@gmail.com,Don Attribution,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,19000,2025-10-28,26,2,2,1,2,0, Neutral  ,1,1,1,SLACK + PLATFORM + MEETING,90,100,GOOD,10df54f90b7e09c4f409ed683aa5d79a,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_pAq25PJR446RfO7vNfRPthR3HPOpWNkRgbbDC3yeXLt,mujeeb_uddin@live.com,Khaja,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,19500,2026-01-14,6,71,6,1,87,0, Neutral  ,1,0,1,SLACK + MEETING,60,70,AVERAGE,0935beb44fe301276400589b94f54be0,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_NPw9sOehmG3RyT70VPmWWjEgpGhaVaPsdv3TM44GcQi,siddhant.navaratna1@gmail.com,Siddhant Navaratna,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,,2024-11-19,915,634,976,,,0,NO_SENTIMENT,,0,,NO DATA,9999,9999,NO DATA,c6dda040158384c0880d68435e2ffb42,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_XdLvff4ZfybY8X3qoAR2C4PYN5VJ8xw00p0k9xQu6Wn,jtcdjs23@gmail.com,Simeon Jones,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,20000,2026-04-01,164,179,163,133,105,0, Positive  ,0,0,0,NO ENGAGEMENT,20,35,POOR,3e0591563f0621cd4b56f2b73af24167,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_SlsTjE2uoy2oI8obdLplJCP8pgO5sGLMaicWIeqdtuU,josueangelsantiago@gmail.com,Josue Angel Santiago,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,,2026-09-19,34,33,45,,,0,NO_SENTIMENT,,0,0,NO DATA,9999,9999,NO DATA,8ce5f3b74e0085d4a6c76cc56fcd5eb3,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_mitMDhSPnOolOaClkxPT8j3BIlLYjuh32Uv0dNupqFS,srinadar@gmail.com,Srinivasa Nadar,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,20000,2026-04-28,51,18,66,8,0,0, Positive  ,1,0,0,SLACK ONLY,30,45,POOR,0eca5f6bafe82a2418e38da7fa66cc89,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_eOWzVQLFV0H4T99WRDtKOnHDpM58L6C7l0N9cxIYvIJ,o_dewan@yahoo.com,Omar Dewan,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,22000,2026-02-23,58,59,76,2,15,0, Positive  ,1,0,0,SLACK ONLY,30,45,POOR,ea402051a841e9eef3975cc8d4b679e4,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_xUaGHejcIKcRKTJNhXtxtcZ9rmCxB7PYMLWPzloJhIe4,roy.mendiola@gmail.com,Roy Mendiola,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,10000,2024-11-19,671,704,736,219,0,0, Positive  ,0,0,,NO DATA,9999,9999,NO DATA,e7cbd182a837c8c1bd05f6f2b4d540cb,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_Cxdbwk0OOI8UZdSlUSFIhUPLbG5NSSkMUJGTJUhvUfj,pkamalkumar@gmail.com,Kamal Potnuru,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,10000,2025-12-17,150,209,11,2,10,0, Positive  ,1,0,1,SLACK + MEETING,60,75,GOOD,f89a4d75f7bd669ab2f3998afafbefae,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_DFUQocqAHFDIA9bFrZbtZLEZXM0aaLAB0gTCQdZo5Yf,pbrousseau74@gmail.com,Peter Rousseau,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,19000,2026-06-28,18,13,18,1,4,0, Neutral  ,1,1,1,SLACK + PLATFORM + MEETING,90,100,GOOD,4bf7a75b9e71eb1dc3af78bccf6bbf7a,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_vSsKQbSrKLkEtonNqv8VBNf5utqPMDs09DnCAVk8aNF,real.tamaratalbott@gmail.com,Tamara Talbott,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,,2024-12-31,159,154,163,106,107,0, Positive  ,0,0,0,NO ENGAGEMENT,20,35,POOR,13b1eb4851240688afb8b825a90935f,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_dRC1pjtWmAtb57UJ5lGyr7oqEw7xOKLNQytliEsi34l,dialloadro@gmail.com / durrah149@outlook.com,abdourahamane diallo,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,5000,2025-09-04,80,424,74,70,4,0, Positive  ,0,0,0,NO ENGAGEMENT,20,35,POOR,0ef1fe47e02e86e0ae20233d392d788a,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
# lead_g4EDJcfm6BqaqNTBYS6zfJM0Ew3vsdKKGgwBIONlj8v,lkrjagadeeshreddy.1995@gmail.com,Jagadeesh,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,11000,2024-11-19,473,10,228,211,20,0, Positive  ,0,1,0,PLATFORM ONLY,50,65,AVERAGE,7a8f61bcd8636e70e0a86f041bb0ca79,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646"""

# t = pd.read_csv(io.StringIO(teacher_txt), skipinitialspace=True)
# t['SENTIMENTS_LAST_30_DAYS'] = t['SENTIMENTS_LAST_30_DAYS'].astype(str).str.strip()

# # Reference conventions: FINAL_SCORE/BASE 9999 means NO DATA -> our NULL
# for c in ['BASE_ENGAGEMENT_SCORE', 'FINAL_SCORE']:
#     t[c] = t[c].where(t[c] != 9999, other=pd.NA)

# ids = ", ".join(f"'{x}'" for x in t['LEAD_ID'])
# ours = spark.sql(f"""
#     SELECT LEAD_ID, CONTRACT_START_DATE, DAYS_SINCE_LAST_LOGIN,
#            SLACK_ENGAGED_FLAG, PLATFORM_ENGAGED_FLAG, MEETING_ENGAGED_FLAG,
#            ENGAGEMENT_PATTERN, BASE_ENGAGEMENT_SCORE, FINAL_SCORE, HEALTH_BAND, VRSN_START_DATE
#     FROM crm_ingestion.gold.customer_health_profile
#     WHERE VRSN_END_DATE IS NULL AND LEAD_ID IN ({ids})
# """).toPandas()

# m = ours.merge(t, on='LEAD_ID', how='left', suffixes=('_ours', '_ref'))
# missing = t[~t['LEAD_ID'].isin(ours['LEAD_ID'])]
# print(f"Compared {len(ours)} of {len(t)} reference leads (missing: {len(missing)})")
# if len(missing):
#     print("Missing from our population: " + ", ".join(
#         f"{r['LEAD_NAME']} ({r['ENGAGEMENT_PATTERN']})" for _, r in missing.iterrows()))

# def same(a, b):
#     # Numeric-aware: spark floats ('0.0') vs reference ints ('0') must compare equal
#     a, b = pd.Series(a), pd.Series(b)
#     both_na = a.isna() & b.isna()
#     an, bn = pd.to_numeric(a, errors='coerce'), pd.to_numeric(b, errors='coerce')
#     num_ok = an.notna() & bn.notna() & (an == bn)
#     str_ok = (a.astype(str).str.strip() == b.astype(str).str.strip()) & ~both_na
#     return both_na | num_ok | str_ok

# # Flags/patterns/scores/bands are drift-free; login date drifts ~4 days (ref snapshot 11-02 vs ours 11-06) -> tolerance 5
# for c in ['SLACK_ENGAGED_FLAG', 'PLATFORM_ENGAGED_FLAG', 'MEETING_ENGAGED_FLAG',
#           'ENGAGEMENT_PATTERN', 'BASE_ENGAGEMENT_SCORE', 'FINAL_SCORE', 'HEALTH_BAND', 'CONTRACT_START_DATE']:
#     m[c + '_match'] = same(m[c + '_ours'], m[c + '_ref'])

# log_o, log_r = m['DAYS_SINCE_LAST_LOGIN_ours'], m['DAYS_SINCE_LAST_LOGIN_ref']
# m['LOGIN_match'] = ((log_o.isna() & log_r.isna())
#                     | ((log_o.notna() & log_r.notna()) & ((log_o - log_r).abs() <= 5)))

# compare_cols = ['CONTRACT_START_DATE', 'DAYS_SINCE_LAST_LOGIN', 'SLACK_ENGAGED_FLAG',
#                 'PLATFORM_ENGAGED_FLAG', 'MEETING_ENGAGED_FLAG', 'ENGAGEMENT_PATTERN',
#                 'BASE_ENGAGEMENT_SCORE', 'FINAL_SCORE', 'HEALTH_BAND']

# print('\n=== MATCH COUNTS (of 19) ===')
# for c in compare_cols:
#     print(f"{c:26s}: {int(m[c + '_match'].sum()) if c != 'DAYS_SINCE_LAST_LOGIN' else int(m['LOGIN_match'].sum())}")

# print('\n=== SIDE-BY-SIDE (key disagreement columns) ===')
# side = pd.DataFrame({
#     'lead': m['LEAD_ID'].str.slice(5, 15),
#     'PATTERN_ours': m['ENGAGEMENT_PATTERN_ours'],
#     'PATTERN_ref': m['ENGAGEMENT_PATTERN_ref'],
#     'PATTERN_match': m['ENGAGEMENT_PATTERN_match'],
#     'PLATFORM_ours': m['PLATFORM_ENGAGED_FLAG_ours'],
#     'PLATFORM_ref': m['PLATFORM_ENGAGED_FLAG_ref'],
#     'PLATFORM_match': m['PLATFORM_ENGAGED_FLAG_match'],
#     'LOGIN_ours': m['DAYS_SINCE_LAST_LOGIN_ours'],
#     'LOGIN_ref': m['DAYS_SINCE_LAST_LOGIN_ref'],
#     'LOGIN_match': m['LOGIN_match'],
#     'CSTART_ours': m['CONTRACT_START_DATE_ours'],
#     'CSTART_ref': m['CONTRACT_START_DATE_ref'],
#     'CSTART_match': m['CONTRACT_START_DATE_match'],
#     'SCORE_ours': m['FINAL_SCORE_ours'],
#     'SCORE_ref': m['FINAL_SCORE_ref'],
#     'SCORE_match': m['FINAL_SCORE_match'],
#     'BAND_match': m['HEALTH_BAND_match'],
# })
# display(side)

# print('\n=== MISMATCH DETAILS ===')
# for c in ['ENGAGEMENT_PATTERN', 'PLATFORM_ENGAGED_FLAG', 'CONTRACT_START_DATE', 'FINAL_SCORE', 'HEALTH_BAND']:
#     bad = m[~m[c + '_match']]
#     if len(bad):
#         print(f"\n-- {c} mismatches ({len(bad)}):")
#         for _, r in bad.iterrows():
#             print(f"  {r['LEAD_ID'][:20]}... ours={r[c + '_ours']}  ref={r[c + '_ref']}")

In [0]:
# # Comparison vs reference sample 2 — the 51 leads pasted in cell 15 (teacher_txt51).
# # Reference window: VRSN 2026-11-01 -> 2026-11-02. Our side: current open versions.
# # Teacher's NO DATA sentinel (9999 scores) normalized to NULL for fair comparison.
# # Clock metrics drift with elapsed days — checked as +4 (Nov 2 snapshot -> Nov 6 ours).
# import io
# import pandas as pd
# import numpy as np

# ref = pd.read_csv(io.StringIO(teacher_txt51))
# ref.columns = [c.strip() for c in ref.columns]

# ids = "','".join(ref["LEAD_ID"])
# ours = spark.sql(f"""
#     SELECT LEAD_ID, CONTRACT_VALUE, CONTRACT_START_DATE,
#            DAYS_SINCE_LAST_EMAIL, DAYS_SINCE_LAST_LOGIN, DAYS_SINCE_LAST_MEETING,
#            SLACK_ENGAGED_FLAG, PLATFORM_ENGAGED_FLAG, MEETING_ENGAGED_FLAG,
#            ENGAGEMENT_PATTERN, BASE_ENGAGEMENT_SCORE, FINAL_SCORE, HEALTH_BAND
#     FROM crm_ingestion.gold.customer_health_profile
#     WHERE VRSN_END_DATE IS NULL AND LEAD_ID IN ('{ids}')
# """).toPandas()

# m = ref.merge(ours, on="LEAD_ID", suffixes=("_ref", "_ours"))
# print(f"reference rows: {len(ref)}   in our population: {len(m)}   missing from ours: {len(ref) - len(m)}")
# missing = ref.loc[~ref["LEAD_ID"].isin(ours["LEAD_ID"]), ["LEAD_ID", "LEAD_NAME", "HEALTH_BAND"]]
# if len(missing):
#     display(missing)

# def num(s):  return pd.to_numeric(s, errors="coerce")
# def asdate(s): return pd.to_datetime(s, errors="coerce")

# m["CONTRACT_START_DATE_ref"]  = asdate(m["CONTRACT_START_DATE_ref"])
# m["CONTRACT_START_DATE_ours"] = asdate(m["CONTRACT_START_DATE_ours"])
# m["CONTRACT_VALUE_ref"]  = num(m["CONTRACT_VALUE_ref"])
# m["CONTRACT_VALUE_ours"] = num(m["CONTRACT_VALUE_ours"])
# for c in ["SLACK_ENGAGED_FLAG", "PLATFORM_ENGAGED_FLAG", "MEETING_ENGAGED_FLAG",
#           "BASE_ENGAGEMENT_SCORE", "FINAL_SCORE"]:
#     m[c + "_ref"]  = num(m[c + "_ref"])
#     m[c + "_ours"] = num(m[c + "_ours"])
# for c in ["BASE_ENGAGEMENT_SCORE_ref", "FINAL_SCORE_ref"]:   # 9999 = his NO DATA sentinel
#     m[c] = m[c].replace(9999, np.nan)
# for c in ["ENGAGEMENT_PATTERN", "HEALTH_BAND"]:
#     m[c + "_ref"]  = m[c + "_ref"].fillna("NO DATA").str.strip()
#     m[c + "_ours"] = m[c + "_ours"].fillna("NO DATA").str.strip()

# def n_match(rc, oc):
#     a, b = m[rc], m[oc]
#     return int(((a.isna() & b.isna()) | (a.notna() & b.notna() & (a == b))).sum())

# print("\n=== MATCH COUNTS (of leads in our population) ===")
# for name, rc, oc in [
#     ("CONTRACT_START_DATE",   "CONTRACT_START_DATE_ref",   "CONTRACT_START_DATE_ours"),
#     ("CONTRACT_VALUE",        "CONTRACT_VALUE_ref",        "CONTRACT_VALUE_ours"),
#     ("SLACK_ENGAGED_FLAG",    "SLACK_ENGAGED_FLAG_ref",    "SLACK_ENGAGED_FLAG_ours"),
#     ("PLATFORM_ENGAGED_FLAG", "PLATFORM_ENGAGED_FLAG_ref", "PLATFORM_ENGAGED_FLAG_ours"),
#     ("MEETING_ENGAGED_FLAG",  "MEETING_ENGAGED_FLAG_ref",  "MEETING_ENGAGED_FLAG_ours"),
#     ("ENGAGEMENT_PATTERN",    "ENGAGEMENT_PATTERN_ref",    "ENGAGEMENT_PATTERN_ours"),
#     ("BASE_ENGAGEMENT_SCORE", "BASE_ENGAGEMENT_SCORE_ref", "BASE_ENGAGEMENT_SCORE_ours"),
#     ("FINAL_SCORE",           "FINAL_SCORE_ref",           "FINAL_SCORE_ours"),
#     ("HEALTH_BAND",           "HEALTH_BAND_ref",           "HEALTH_BAND_ours"),
# ]:
#     print(f"{name:<22}: {n_match(rc, oc)}/{len(m)}")

# # headline 1 — contract-start mismatches side by side
# cs = m[m["CONTRACT_START_DATE_ref"].astype(str) != m["CONTRACT_START_DATE_ours"].astype(str)]
# print(f"\nCONTRACT_START_DATE mismatches: {len(cs)}")
# if len(cs):
#     display(cs[["LEAD_ID", "LEAD_NAME", "CONTRACT_START_DATE_ref", "CONTRACT_START_DATE_ours"]])

# # headline 2 — overlap with our 37 contract-gap leads
# gap_ids = spark.sql("""
#     SELECT LEAD_ID FROM crm_ingestion.gold.customer_health_profile
#     WHERE VRSN_END_DATE IS NULL AND CONTRACT_START_DATE IS NULL
# """).toPandas()["LEAD_ID"].tolist()
# ov = m[m["LEAD_ID"].isin(gap_ids)]
# print(f"of our 37 contract-gap leads, present in this sample: {len(ov)}")
# if len(ov):
#     display(ov[["LEAD_ID", "LEAD_NAME", "CONTRACT_START_DATE_ref", "CONTRACT_START_DATE_ours", "HEALTH_BAND_ref"]])

# # clock drift — his snapshot closed 2026-11-02; ours is 2026-11-06 (+4 if nothing new)
# d = m["DAYS_SINCE_LAST_LOGIN_ours"] - m["DAYS_SINCE_LAST_LOGIN_ref"]
# print(f"login clock: +4 days exactly: {(d == 4).sum()}, lower (new login since snapshot): {(d < 0).sum()}, other: {((d != 4) & (d >= 0)).sum()} of {len(m)}")